STEP 1. SETUP · imports & folders

In [1]:
# ─── 1 · SETUP ──────────────────────────────────────────────────────
import os
import glob
import zipfile
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from rasterstats import zonal_stats
import matplotlib.pyplot as plt

# Paths
WORK        = Path(r"C:\valhalla_neat")
TRIP_DIR    = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data"
)
UTCI_DIR    = Path(
    r"C:\Users\Agustin\Documents\2025summer\trees\top1000\nyc_utci1000_maps"
)
RESULTS_DIR = Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("→ outputs →", RESULTS_DIR)


→ outputs → C:\Users\Agustin\Documents\2025summer\trees\top1000\results


STEP 2 · load all trips & collapse station coordinates

In [2]:
# ─── 2 · LOAD TRIPS & COLLAPSE COORDS  (reads *every* CSV in every ZIP) ─────────
import zipfile
import pandas as pd
from pathlib import Path

# Path to your trip‐data folder (June 2024 ZIP is in there).
TRIP_DIR = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data"
)

# Include 'ride_id' so we carry the true trip identifier downstream
USE = [
    "ride_id",
    "started_at","start_station_id","end_station_id",
    "start_lat","start_lng","end_lat","end_lng"
]

# Force floats to float32; strings to “string”
DTYPE = {c: "float32" for c in USE if c.endswith(("_lat","_lng"))}
DTYPE.update({
    "ride_id":            "string",
    "started_at":         "string",
    "start_station_id":   "string",
    "end_station_id":     "string"
})

def read_trips(zp: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zp) as z:
        csv_parts = [f for f in z.namelist() if f.endswith(".csv")]
        frames = []
        for csv_fname in csv_parts:
            df_part = pd.read_csv(
                z.open(csv_fname),
                usecols=USE,
                dtype=DTYPE,
                low_memory=False
            )
            frames.append(df_part)
    return pd.concat(frames, ignore_index=True)

# Gather and list all monthly ZIPs
zip_paths = sorted(TRIP_DIR.glob("*.zip"))
print(f"► found {len(zip_paths)} ZIP archive(s) in trip_data:")
for z in zip_paths:
    print("   •", z.name)

# Read each ZIP and concatenate
frames = []
for z in zip_paths:
    df_z = read_trips(z)
    print(f"{z.name:35s} → {len(df_z):,} rows before dropna")
    frames.append(df_z)

trips = pd.concat(frames, ignore_index=True)
print(f"\n► TOTAL TRIPS LOADED: {len(trips):,} rows before dropna\n")

# Clean & drop
trips.replace(r"^\s*$", pd.NA, regex=True, inplace=True)
pre = len(trips)
trips.dropna(inplace=True)
post = len(trips)
print(f"► AFTER dropna: {post:,} rows remain  (dropped {pre-post:,})\n")

# Convert started_at to datetime
trips["started_at"] = pd.to_datetime(trips["started_at"])

# Compute canonical station coordinates
stations = (
    pd.concat([
        trips[["start_station_id","start_lat","start_lng"]]
             .rename(columns={
                 "start_station_id":"sid",
                 "start_lat":"lat","start_lng":"lon"
             }),
        trips[["end_station_id","end_lat","end_lng"]]
             .rename(columns={
                 "end_station_id":"sid",
                 "end_lat":"lat","end_lng":"lon"
             })
    ])
    .groupby("sid").median().reset_index()
)

lat_map = stations.set_index("sid")["lat"]
lon_map = stations.set_index("sid")["lon"]

for col, mapper in [
    ("start_lat", lat_map),
    ("start_lng", lon_map),
    ("end_lat",   lat_map),
    ("end_lng",   lon_map)
]:
    trips[col] = trips[col
                       .replace("_lat","_station_id")
                       .replace("_lng","_station_id")
                      ].map(mapper)

print(f"► {len(stations):,} unique stations after coordinate collapse")


► found 1 ZIP archive(s) in trip_data:
   • 202406-citibike-tripdata.zip
202406-citibike-tripdata.zip        → 4,783,576 rows before dropna

► TOTAL TRIPS LOADED: 4,783,576 rows before dropna

► AFTER dropna: 4,768,748 rows remain  (dropped 14,828)

► 2,309 unique stations after coordinate collapse


STEP 3. load pre-computed NEAT routes

In [3]:
# ─── 3 · LOAD PRE-COMPUTED ROUTES ──────────────────────────────────
route_gpkg = WORK / "verified_station_pair_routes_fixed.gpkg"
routes = (
    gpd.read_file(route_gpkg, layer="routes_with_correct_ids")
       .to_crs(4326)[["orig_start","orig_end","geometry"]]
)
print("routes table loaded:", len(routes))


routes table loaded: 738979


STEP 4. load NEAT street geometry (no UTCI yet)

In [4]:
# ─── 4 · LOAD NEAT STREET GEOMETRY ─────────────────────────────────
streets_base = gpd.read_file(
    r"C:\Users\Agustin\Documents\2025summer\neatnet_generator\verified_streets_neat_bidirectional.shp"
).to_crs(32118)

streets_base["seg_id"] = streets_base.index
street_len = streets_base.length
print("NEAT street geometry:", len(streets_base))


NEAT street geometry: 58472


STEP 5. D-HELPERS · updated (sets index name to None)

In [5]:
# ─── 5 · HELPERS (with neutral index name) ─────────────────────────
BUF, MIN_LEN, MIN_RATIO, HOT = 3, 10, 0.20, 32.0

def utci_path(hour):
    if 0 <= hour <= 20:
        return UTCI_DIR / f"nyc_utci1000_173_{hour:02d}00.tif"
    elif hour == 21:
        return UTCI_DIR / "nyc_utci1000_172_2100.tif"
    elif hour == 22:
        return UTCI_DIR / "nyc_utci1000_172_2200.tif"
    elif hour == 23:
        return UTCI_DIR / "nyc_utci1000_172_2300.tif"
    else:
        raise ValueError("hour out of range")

def attach_routes(df):
    """
    Join each trip to its route geometry using the original station IDs
    (orig_start, orig_end). We assume `routes` has columns:
      orig_start, orig_end, geometry
    """
    return df.merge(
        routes[["orig_start","orig_end","geometry"]],
        left_on=["start_station_id","end_station_id"],
        right_on=["orig_start","orig_end"],
        how="left"
    ).dropna(subset=["geometry"])

def street_utci(hour):
    tif = utci_path(hour)
    cache = RESULTS_DIR / f"street_segments_utci_neat_{hour:02d}.gpkg"
    if cache.exists():
        return gpd.read_file(cache).set_index("seg_id")
    s = streets_base.copy()
    print(f"    computing UTCI zonal stats for {tif.name} …")
    zs = zonal_stats(s.geometry, tif, stats=["mean"],
                     all_touched=True, nodata=None)
    s["utci"] = [z["mean"] if z["mean"] is not None else np.nan for z in zs]
    s.to_file(cache, driver="GPKG")
    return s.set_index("seg_id")


STEP 6. GENERATING THE AVERAGE UTCI FOR STREETS SEGMENTS FOR ALL HOURS

In [6]:
# ─── 6 · GENERATE UTCI‐ANNOTATED NEAT STREETS FOR ALL HOURS ────────────
# Uses your street_utci(hour) helper to produce one GPKG per hour if missing.
for hh in range(24):
    cache = RESULTS_DIR / f"street_segments_utci_neat_{hh:02d}.gpkg"
    if not cache.exists():
        print(f"⏳ Generating {cache.name} …")
        # this will read your simplified shapefile, run zonal_stats on the hh:00 TIFF,
        # attach mean utci values, set seg_id, and write out the GPKG
        street_utci(hh)
    else:
        print(f"✅ Already have {cache.name}")


⏳ Generating street_segments_utci_neat_00.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0000.tif …
⏳ Generating street_segments_utci_neat_01.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0100.tif …
⏳ Generating street_segments_utci_neat_02.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0200.tif …
⏳ Generating street_segments_utci_neat_03.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0300.tif …
⏳ Generating street_segments_utci_neat_04.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0400.tif …
⏳ Generating street_segments_utci_neat_05.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0500.tif …
⏳ Generating street_segments_utci_neat_06.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0600.tif …
⏳ Generating street_segments_utci_neat_07.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_0700.tif …
⏳ Generating street_segments_utci_neat_08.gpkg …
    computing UTCI zonal stats for nyc_utci1000_173_080

RUN PYTHON SCRIPT "C:\Users\Agustin\Documents\2025summer\trees\top1000\build_hotspots_1000trees.py"

STEP 7. to create
"street_network1000trees_with_counts_and_length.gpkg"

In [1]:
import geopandas as gpd
from pathlib import Path

# Paths
INPUT_FP  = Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results\street_network_1000trees_with_counts.gpkg")
OUTPUT_FP = INPUT_FP.parent / "street_network1000trees_with_counts_and_length.gpkg"

# Load and reproject (ensures geometry length is in meters)
gdf = gpd.read_file(INPUT_FP).to_crs(32118)

# Compute segment length in meters
gdf["length_m"] = gdf.geometry.length

# Drop the utci column
if "utci" in gdf.columns:
    gdf = gdf.drop(columns=["utci"])

# Select and order columns: seg_id, count, length_m, then geometry
cols = ["seg_id", "count", "length_m", "geometry"]
gdf = gdf[cols]

# Save new GeoPackage
gdf.to_file(OUTPUT_FP, driver="GPKG")
print("Wrote:", OUTPUT_FP)


Wrote: C:\Users\Agustin\Documents\2025summer\trees\top1000\results\street_network1000trees_with_counts_and_length.gpkg


STEP 8. GENERATE TOP150, 400, 1000

In [2]:
import geopandas as gpd
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────
DIR        = Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results")
FULL_NET   = DIR / "street_network_1000trees_with_counts.gpkg"
TARGETS    = [150, 400, 1000]  # how many top segments to export

# Segment IDs to exclude
exclude_ids = {58251, 58267, 27, 53086, 56194, 58345, 10148, 53324, 9712, 62, 70, 58260, 12, 52736, 29566}

# ─── LOAD FULL NETWORK WITH COUNTS ────────────────────────────────────
net = gpd.read_file(FULL_NET).set_index("seg_id")
print(f"Loaded full network: {len(net)} segments")

# ─── EXCLUDE SPECIFIED SEGMENTS ───────────────────────────────────────
net_filtered = net.drop(labels=exclude_ids, errors="ignore")
print(f"After exclusion: {len(net_filtered)} segments (removed {len(net) - len(net_filtered)} if present)")

# ─── SELECT & WRITE MULTIPLE TOP-N SETS ───────────────────────────────
for n in TARGETS:
    topN = net_filtered.nlargest(n, "count")
    print(f"Selected top {n} segments: {len(topN)}")
    out_fp = DIR / f"top{n}_high_utci_segments_1000treesGLOBAL.gpkg"
    topN.reset_index().to_file(out_fp, driver="GPKG")
    print("Wrote:", out_fp)


Loaded full network: 58472 segments
After exclusion: 58457 segments (removed 15 if present)
Selected top 150 segments: 150
Wrote: C:\Users\Agustin\Documents\2025summer\trees\top1000\results\top150_high_utci_segments_1000treesGLOBAL.gpkg
Selected top 400 segments: 400
Wrote: C:\Users\Agustin\Documents\2025summer\trees\top1000\results\top400_high_utci_segments_1000treesGLOBAL.gpkg
Selected top 1000 segments: 1000
Wrote: C:\Users\Agustin\Documents\2025summer\trees\top1000\results\top1000_high_utci_segments_1000treesGLOBAL.gpkg


STEP 9. Fill in the missing trip_len_m in your cooled CSV by copying from the base case

In [1]:
import pandas as pd
from pathlib import Path

# === Directories ===
base_dir   = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\utci_results")            # BASE lengths (trusted)
target_dir = Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\utci_results")           # Top-1000 CSVs to fix

# === Process each hour ===
for hh in range(24):
    hour_str = f"{hh:02d}00"
    base_fp   = base_dir   / f"mean_utci_{hour_str}.csv"
    target_fp = target_dir / f"mean_utci_{hour_str}.csv"

    if not base_fp.exists() or not target_fp.exists():
        print(f"⏭️ Skipping {hour_str}: missing file in base or target.")
        continue

    # Load base lengths and target UTCI rows
    df_base   = pd.read_csv(base_fp, usecols=["ride_id", "trip_len_m"]).rename(columns={"trip_len_m": "trip_len_m_base"})
    df_target = pd.read_csv(target_fp)

    if "ride_id" not in df_target.columns:
        print(f"⚠️ {hour_str}: target CSV missing 'ride_id' column — cannot merge.")
        continue

    # Merge and replace lengths
    cols_order = df_target.columns.tolist()
    merged = df_target.merge(df_base, on="ride_id", how="left")

    # Report any ride_ids that didn’t find a base match
    missing = merged["trip_len_m_base"].isna().sum()
    if missing:
        print(f"⚠️ {hour_str}: {missing} ride_id(s) missing in base file; keeping target lengths for those.")
        # keep original lengths where base is missing
        merged["trip_len_m"] = merged["trip_len_m_base"].fillna(merged["trip_len_m"])
    else:
        merged["trip_len_m"] = merged["trip_len_m_base"]

    # Restore original column order (drop helper col)
    if "trip_len_m_base" in merged.columns:
        merged = merged.drop(columns=["trip_len_m_base"])
    merged = merged[cols_order]

    # Write back
    merged.to_csv(target_fp, index=False)
    total_len = pd.to_numeric(merged["trip_len_m"], errors="coerce").sum()
    print(f"✔ {hour_str}: updated. total trip_len_m = {total_len:,.0f} m")


✔ 0000: updated. total trip_len_m = 229,517,472 m
✔ 0100: updated. total trip_len_m = 137,180,049 m
✔ 0200: updated. total trip_len_m = 90,774,267 m
✔ 0300: updated. total trip_len_m = 56,927,940 m
✔ 0400: updated. total trip_len_m = 48,889,772 m
✔ 0500: updated. total trip_len_m = 90,407,647 m
✔ 0600: updated. total trip_len_m = 246,352,546 m
✔ 0700: updated. total trip_len_m = 477,609,820 m
✔ 0800: updated. total trip_len_m = 725,690,274 m
✔ 0900: updated. total trip_len_m = 610,521,390 m
✔ 1000: updated. total trip_len_m = 550,984,627 m
✔ 1100: updated. total trip_len_m = 607,482,140 m
✔ 1200: updated. total trip_len_m = 668,818,534 m
✔ 1300: updated. total trip_len_m = 710,441,365 m
✔ 1400: updated. total trip_len_m = 775,581,604 m
✔ 1500: updated. total trip_len_m = 842,903,875 m
✔ 1600: updated. total trip_len_m = 931,359,678 m
✔ 1700: updated. total trip_len_m = 1,215,752,827 m
✔ 1800: updated. total trip_len_m = 1,140,248,088 m
✔ 1900: updated. total trip_len_m = 906,140,001 m
